<a href="https://www.kaggle.com/code/asivakumarnair/diabetic-retinopathy-imagenet?scriptVersionId=343771281" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== FULL REBUILD, NEW SESSION: environment through Stage 11, MESSIDOR, MOBILENETV2 + RESNET50 =====
# LAST CELL OF STAGE 11. Once this completes, all 12 single-source models are done:
# APTOS (4/4), EyePACS (4/4), Messidor (4/4).

!pip install -q tensorflow==2.19.0

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

# ---------- CONFIG ----------
MESSIDOR_CSV  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor_data.csv'
MESSIDOR_IMG  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor-2/messidor-2/preprocess'

GRADES      = ['0','1','2','3','4']
IMG_SIZE, BATCH_SIZE = 224, 32
PHASE1_EPOCHS, PHASE1_LR, PHASE2_LR, EARLYSTOP_PAT, MONITOR = 10, 1e-3, 1e-5, 7, 'val_accuracy'
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)

# ---------- DATA REBUILD, MESSIDOR ONLY ----------
messidor = pd.read_csv(MESSIDOR_CSV)
messidor['grade']      = messidor['diagnosis'].astype(int).astype(str)
messidor['image_path'] = MESSIDOR_IMG + '/' + messidor['id_code'].astype(str)
messidor['source']     = 'messidor'
messidor['patient_id'] = None

is_im = ~messidor['image_path'].str.contains(r'\d{8}_\d+_\d+_PP\.png$', regex=True)
im_check = messidor[is_im].copy()
im_check['im_num'] = im_check['image_path'].str.extract(r'IM(\d+)\.JPG$').astype(int)
im_check = im_check.sort_values('im_num').reset_index(drop=True)
im_check['pid'] = im_check.index // 2
sizes = im_check.groupby('pid').size()
full_pairs = im_check[im_check['pid'].isin(sizes[sizes == 2].index)]
agreement = full_pairs.groupby('pid')['grade'].apply(lambda g: g.iloc[0] == g.iloc[1])
print(f"Pairing agreement: {agreement.mean():.3f} (expect ~0.749)")
assert abs(agreement.mean() - 0.749) < 0.01, "Pairing evidence did not reproduce, stop and investigate"

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, falling back to unstratified.")
        print(f"  sklearn error: {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_image_level(df, rs=SEED, tag=""):
    tr, tmp = safe_split(df, 'grade', 0.30, rs, tag=f"{tag} first")
    va, te  = safe_split(tmp, 'grade', 0.50, rs, tag=f"{tag} second")
    return tr, va, te

def split_messidor_mixed(df, rs=SEED, tag="Messidor"):
    is_im = ~df['image_path'].str.contains(r'\d{8}_\d+_\d+_PP\.png$', regex=True)
    im_df = df[is_im].copy()
    im_df['im_num'] = im_df['image_path'].str.extract(r'IM(\d+)\.JPG$').astype(int)
    im_df = im_df.sort_values('im_num').reset_index(drop=True)
    im_df['patient_id'] = 'messidor_pair_' + (im_df.index // 2).astype(str)
    pg = im_df.groupby('patient_id')['grade'].max().reset_index()
    p_tr, p_tmp = safe_split(pg, 'grade', 0.30, rs, tag=f"{tag} IM first")
    p_va, p_te  = safe_split(p_tmp, 'grade', 0.50, rs, tag=f"{tag} IM second")
    pick = lambda ids: im_df[im_df['patient_id'].isin(ids['patient_id'])]
    im_tr, im_va, im_te = pick(p_tr), pick(p_va), pick(p_te)
    s_tr, s_va, s_te = set(p_tr['patient_id']), set(p_va['patient_id']), set(p_te['patient_id'])
    assert s_tr.isdisjoint(s_va) and s_tr.isdisjoint(s_te) and s_va.isdisjoint(s_te), "MESSIDOR IM PATIENT LEAKAGE"
    print(f"{tag} IM-style patient-leakage check: PASS ({len(im_df)} images, {im_df['patient_id'].nunique()} groups)")
    date_df = df[~is_im]
    d_tr, d_va, d_te = split_image_level(date_df, rs, tag=f"{tag} date-style")
    cat = lambda a, b: pd.concat([a.drop(columns=['im_num']), b], ignore_index=True)
    return cat(im_tr, d_tr), cat(im_va, d_va), cat(im_te, d_te)

m_tr, m_va, m_te = split_messidor_mixed(messidor)
print(f"\nMessidor split: Train {len(m_tr)} | Val {len(m_va)} | Test {len(m_te)}")

cls = np.array(GRADES)
cw = compute_class_weight('balanced', classes=cls, y=m_tr['grade'])
messidor_class_weight = {i: w for i, w in enumerate(cw)}
span = cw.max()/cw.min()
print(f"\nMessidor class weight span: {span:.1f}x (expect ~31.0x)")

# ================================================================
# STAGE 11: MESSIDOR, MOBILENETV2 then RESNET50 (final two cells of Stage 11)
# ================================================================

def make_source_gens(preprocess_fn, tr_df, va_df, te_df):
    train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
    eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='grade', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=GRADES, color_mode='rgb')
    tr = train_idg.flow_from_dataframe(tr_df, shuffle=True,  seed=SEED, **common)
    va = eval_idg.flow_from_dataframe(va_df,  shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(te_df,  shuffle=False, **common)
    return tr, va, te

def build_pretrained(base_class, num_classes=5, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(),
                         Dense(256,activation='relu'), Dropout(0.3),
                         Dense(num_classes,activation='softmax')])
    return model, base

def train_source_pretrained(base_class, preprocess_fn, arch_code, source_name, tr_df, va_df, te_df, class_weight):
    tag_p1 = f"ss_{arch_code}_{source_name}_dr_phase1"
    tag_p2 = f"ss_{arch_code}_{source_name}_dr"
    tr, va, te = make_source_gens(preprocess_fn, tr_df, va_df, te_df)
    model, base = build_pretrained(base_class)

    base.trainable = False
    model.compile(Adam(PHASE1_LR), 'categorical_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
    print(f"\n===== {arch_code}, {source_name}: PHASE 1 (head only, {PHASE1_EPOCHS} epochs) =====")
    model.fit(tr, validation_data=va, epochs=PHASE1_EPOCHS, class_weight=class_weight,
              callbacks=[ModelCheckpoint(f'/kaggle/working/{tag_p1}.keras', monitor=MONITOR, save_best_only=True),
                         CSVLogger(f'/kaggle/working/{tag_p1}_log.csv', append=False)], verbose=1)
    print(f"{arch_code}, {source_name} Phase 1 saved.")

    base.trainable = True
    model.compile(Adam(PHASE2_LR), 'categorical_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
    print(f"\n===== {arch_code}, {source_name}: PHASE 2 (full fine-tune, up to 60 epochs) =====")
    model.fit(tr, validation_data=va, epochs=60, class_weight=class_weight,
              callbacks=[EarlyStopping(monitor=MONITOR, patience=EARLYSTOP_PAT, restore_best_weights=True),
                         ModelCheckpoint(f'/kaggle/working/{tag_p2}.keras', monitor=MONITOR, save_best_only=True),
                         CSVLogger(f'/kaggle/working/{tag_p2}_log.csv', append=False)], verbose=1)
    result = model.evaluate(te, verbose=0)
    print(f"\n{tag_p2} TEST: loss={result[0]:.4f} accuracy={result[1]:.4f} auc={result[2]:.4f}")
    print(f"{arch_code}, {source_name} Phase 2 saved.")
    return {'arch':arch_code,'source':source_name,'loss':result[0],'accuracy':result[1],'auc':result[2]}

stage11_results = []

r = train_source_pretrained(MobileNetV2, mob_pre, 'mob', 'messidor', m_tr, m_va, m_te, messidor_class_weight)
stage11_results.append(r)
pd.DataFrame(stage11_results).to_csv('/kaggle/working/dr_stage11_messidor_mob.csv', index=False)
print("\nCheckpointed after MobileNetV2.")
tf.keras.backend.clear_session()

r = train_source_pretrained(ResNet50, res_pre, 'res', 'messidor', m_tr, m_va, m_te, messidor_class_weight)
stage11_results.append(r)
pd.DataFrame(stage11_results).to_csv('/kaggle/working/dr_stage11_messidor_mob_res.csv', index=False)
print("\nCheckpointed after ResNet50.")

print("\n===== MESSIDOR, MobileNetV2 + ResNet50, COMPLETE =====")
print(pd.DataFrame(stage11_results).to_string(index=False))
print("\n" + "="*60)
print("STAGE 11 FULLY COMPLETE: all 12 single-source models trained.")
print("APTOS (4/4), EyePACS (4/4), Messidor (4/4)")
print("Next: Stage 12, cross-source evaluation matrix (inference only, no training)")
print("="*60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 27.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.0 which is incompatible.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.


2026-08-20 18:30:13.089216: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787250613.112794      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787250613.119808      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787250613.138384      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787250613.138403      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787250613.138405      24 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
Pairing agreement: 0.749 (expect ~0.749)
WARNING [Messidor IM second]: stratified split failed, falling back to unstratified.
  sklearn error: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.
Messidor IM-style patient-leakage check: PASS (687 images, 344 groups)

Messidor split: Train 1219 | Val 262 | Test 263

Messidor class weight span: 31.0x (expect ~31.0x)
Found 1219 validated image filenames belonging to 5 classes.
Found 262 validated image filenames belonging to 5 classes.
Found 263 validated image filenames belonging to 5 classes.


I0000 00:00:1787250627.493013      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787250627.499158      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 [==============================] - 2s 0us/step

===== mob, messidor: PHASE 1 (head only, 10 epochs) =====
Epoch 1/10


I0000 00:00:1787250635.969410      76 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1787250638.476315      74 service.cc:152] XLA service 0x7968c530e490 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787250638.476347      74 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1787250638.476351      74 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1787250638.628190      74 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


39/39 [==============================] - 42s 929ms/step - loss: 1.9252 - accuracy: 0.2436 - auc: 0.6103 - val_loss: 1.3503 - val_accuracy: 0.2443 - val_auc: 0.6951
Epoch 2/10
39/39 [==============================] - 22s 557ms/step - loss: 1.4770 - accuracy: 0.3486 - auc: 0.7244 - val_loss: 1.3208 - val_accuracy: 0.3321 - val_auc: 0.7233
Epoch 3/10
39/39 [==============================] - 22s 560ms/step - loss: 1.2532 - accuracy: 0.3856 - auc: 0.7541 - val_loss: 1.1387 - val_accuracy: 0.4084 - val_auc: 0.8100
Epoch 4/10
39/39 [==============================] - 22s 562ms/step - loss: 1.2732 - accuracy: 0.3675 - auc: 0.7435 - val_loss: 1.0699 - val_accuracy: 0.5267 - val_auc: 0.8408
Epoch 5/10
39/39 [==============================] - 21s 547ms/step - loss: 1.1835 - accuracy: 0.4085 - auc: 0.7828 - val_loss: 1.2013 - val_accuracy: 0.4275 - val_auc: 0.7850
Epoch 6/10
39/39 [==============================] - 21s 550ms/step - loss: 1.1524 - accuracy: 0.4717 - auc: 0.8101 - val_loss: 1.6541 - 